# Using AI agents: first in VS Code, then inside Docker

This notebook is a gentle introduction. You do **not** need to know anything about Docker,
containers or software engineering to follow it. Every new word is explained the first time it
appears, and there is a short glossary at the end.

**What you will do**

| Stage | What happens | Why |
|---|---|---|
| **1. Agent in VS Code** | Install an AI agent on your own computer and give it a small materials-science task. | See what an agent is and what it does. |
| **2. Agent inside Docker** | Run the same agent inside an isolated "box" on your computer. | Keep the agent away from your personal files, and get a setup anyone can copy exactly. |
| **3. Next steps** *(optional)* | See what an agent *cannot* do on its own, and how tools fix it. | A preview of the advanced notebook. |

We use two agents: **Claude Code** (made by Anthropic) and **Gemini CLI** (made by Google). You can
follow along with just one of them.

**How to read this notebook**

- 🖥️ **Terminal** boxes: text you type into a terminal (explained in 1.2).
- 📄 **File** boxes: a file you create, with the name given above the box.
- ▶️ **Code cells** (the grey boxes with a *Run* button): small checks. Run them to confirm each
  step worked before moving on.

---
# Stage 1 — An agent in VS Code

## 1.1 What is an "agent"?

You have probably used a chatbot like ChatGPT, Claude or Gemini in a web browser. You type a
question, it types an answer. That is all it can do: **it can only write text.** It cannot open
your files, run your Python script or check whether its answer is correct.

An **agent** is the same kind of AI, but connected to your computer so that it can *act*:

```
   you give a task
         │
         ▼
 ┌──────────────────┐     "I will read the file"      ┌─────────────────────┐
 │  the AI (model)  │ ──────────────────────────────▶ │  your computer runs │
 │  decides the     │                                 │  that action        │
 │  next step       │ ◀────────────────────────────── │                     │
 └──────────────────┘     the real result (or error)  └─────────────────────┘
         │
         ▼   repeats until the task is done
```

Each round, the AI chooses an action — *read a file*, *write a file*, *run a command* — the
program actually does it, and the AI sees the real result. If its Python script crashes, it sees
the real error message and can fix it. That loop is the whole difference between a chatbot and an
agent.

Each of these actions is called a **tool call**. Remember that name: when you want to know whether
to trust an agent's answer, look at which tool calls it made to get there.

## 1.2 Install the basics

You need three programs. If you already have one, skip it.

**a) VS Code** — a free code editor. Download it from
[code.visualstudio.com](https://code.visualstudio.com/) and install it normally.

**b) Python** — to run the check cells in this notebook. Download from
[python.org](https://www.python.org/downloads/) (on Windows, tick *"Add Python to PATH"* in the
installer). Then, in VS Code, open the **Extensions** panel (the icon with four squares on the left
bar) and install **Python** and **Jupyter**, both by Microsoft.

**c) Node.js** — some programs are written in JavaScript and need Node.js to run, in the same way a
`.py` file needs Python. Gemini CLI is one of them. Download the **LTS** version from
[nodejs.org](https://nodejs.org/).

### What is a terminal?

A **terminal** is a window where you type commands instead of clicking buttons. VS Code has one
built in: menu **Terminal → New Terminal**, or press `` Ctrl+` `` (the key above Tab).
You type a command, press Enter, and the result is printed below.

Check that everything is installed by typing these in the terminal, one at a time:

🖥️ Terminal
```bash
python --version
node --version
npm --version
```

Each should print a version number. (`npm` came with Node.js — it is the tool that installs
JavaScript programs, just like `pip` installs Python packages.) On macOS/Linux, if `python` is not
found, try `python3`.

## 1.3 Install the agents

**Claude Code**

🖥️ Terminal — macOS / Linux
```bash
curl -fsSL https://claude.ai/install.sh | bash
```

🖥️ Terminal — Windows (PowerShell, the default VS Code terminal on Windows)
```powershell
irm https://claude.ai/install.ps1 | iex
```

You can also install the **Claude Code** extension (by Anthropic) from the VS Code Extensions
panel. It adds a Claude panel inside the editor. It is optional: everything in this notebook works
from the terminal.

**Gemini CLI**

🖥️ Terminal
```bash
npm install -g @google/gemini-cli
```

(`-g` means "global": install it for the whole computer, not just this folder.)

**Close the terminal and open a new one** so it notices the new programs. Then check:

🖥️ Terminal
```bash
claude --version
gemini --version
```

> **"CLI"** in *Gemini CLI* means *command-line interface*: a program you use by typing in a
> terminal. Both agents here work that way.

## 1.4 Make a project folder and sign in

Create an empty folder for this tutorial and open it in VS Code:

🖥️ Terminal
```bash
mkdir agents-demo
cd agents-demo
code .
```

(`mkdir` = make a folder, `cd` = go into it, `code .` = open the current folder in VS Code.
If `code` is not found, just use **File → Open Folder…** instead.)

Put this notebook inside that folder and open it from there. When you run the first code cell, VS
Code asks you to **select a kernel** — the kernel is simply *the Python that runs the cells*.
Choose the Python you installed in 1.2.

Now sign in to each agent. In the VS Code terminal:

🖥️ Terminal
```bash
claude
```

A browser page opens. Log in with your Claude account (Pro, Max, Team or a Console account).
Type `/exit` to leave.

🖥️ Terminal
```bash
gemini
```

Choose **Sign in with Google** and log in with any Google account. Press `Ctrl+C` twice to leave.

You only do this once. Run the cell below to check that everything is found.

In [ ]:
import shutil, sys

print("Python:", sys.version.split()[0])
for program in ("claude", "gemini", "node", "npm"):
    where = shutil.which(program)
    print(f"{program:7} ->", where if where else "NOT FOUND (see 1.2 / 1.3)")

## 1.5 A first task

We will give the agent a small problem whose answer we already know, so we can check it.

Run the cell below. It creates a **POSCAR** file (the VASP structure format) for an idealised
rock-salt NiO crystal with lattice constant a = 4.17 Å. In rock-salt, the nearest Ni–O distance is
exactly a/2, so the correct answer is easy to check by hand.

In [ ]:
from pathlib import Path

POSCAR = """Idealised rock-salt NiO (teaching file, a = 4.17 A)
1.0
   4.17  0.00  0.00
   0.00  4.17  0.00
   0.00  0.00  4.17
Ni O
4 4
Direct
   0.0 0.0 0.0
   0.0 0.5 0.5
   0.5 0.0 0.5
   0.5 0.5 0.0
   0.5 0.5 0.5
   0.5 0.0 0.0
   0.0 0.5 0.0
   0.0 0.0 0.5
"""

Path("structures").mkdir(exist_ok=True)
Path("structures/NiO_POSCAR").write_text(POSCAR)
print("Created structures/NiO_POSCAR")
print("Correct answer: nearest Ni-O distance = 4.17 / 2 =", 4.17 / 2, "A, with 6 neighbours")

Now start an agent in the terminal (`claude` or `gemini`) and paste this request:

```
Read structures/NiO_POSCAR. Write a short Python script that finds every Ni-O
distance shorter than 3 A, including neighbouring periodic cells, and run it.
Tell me the nearest-neighbour distance and how many neighbours are at that
distance. Use only standard Python, don't install anything.
```

**What you will see**

1. The agent says what it plans to do.
2. It **asks your permission** before writing a file or running a command. Read what it wants to
   do, then approve. This is your main safety control — don't approve things you don't understand.
3. It reads the file, writes a script, runs it and reads the output.
4. If the script fails, it sees the error and fixes it by itself.

**Check the answer.** It should be **2.085 Å with 6 neighbours** (octahedral coordination). If the
agent says something else, it made a mistake — and you caught it because you knew what to expect.

Try the same request with the other agent and compare: how many steps did each take? Did either
make an error and recover?

In [ ]:
# Which files did the agent create?
from pathlib import Path

for f in sorted(Path(".").glob("*.py")) + sorted(Path("structures").glob("*")):
    print(f"{f.stat().st_size:>6} bytes  {f}")
print("\nOpen the script the agent wrote and read it. You are allowed to disagree with it.")

## 1.6 What can the agent reach?

The agent runs **as you**, on **your computer**. It normally works in the folder you opened, and it
asks before doing anything — but the commands it runs have the same access you have: your home
folder, your documents, your SSH keys, your cluster login.

Most of the time that is fine. But two situations make people uneasy:

- **Mistakes.** You approve a command too quickly, and it deletes or changes the wrong files.
- **Sharing.** A colleague wants the same setup. "Install Python, then Node, then these packages…"
  rarely produces exactly the same result on two computers.

Both problems have the same solution: run the agent inside a **container**. That is Stage 2.

---
# Stage 2 — The same agent, inside Docker

## 2.1 Docker in plain words

Imagine a **separate, disposable computer that lives inside your computer**. It has its own
folders and its own installed programs. The agent works inside it and can only see what you put
there. When you are done you throw it away, and your real computer is exactly as it was.

That "computer inside your computer" is a **container**, and **Docker** is the program that creates
and runs containers. Only four words matter here:

| Word | Plain meaning | Everyday analogy |
|---|---|---|
| **Docker** | The program that builds and runs containers. | The oven. |
| **Image** | A saved recipe + starting point: an operating system and a list of software. | The recipe. |
| **Container** | A running copy made from an image. You can make many; each is isolated. | The cake. Bake a new one any time. |
| **Dev container** | A container that VS Code opens your project folder *inside*, so the terminal, the agent and the notebook all run in the container. | — |

Two more facts that will matter later:

- **Your project folder is shared.** The folder you open in VS Code is visible inside the container
  (at `/workspaces/agents-demo`). Files the agent writes there appear on your computer too. Nothing
  else from your computer is visible inside.
- **Rebuilding erases the container.** Anything saved inside the container but *outside* the project
  folder is lost when you rebuild it. We will deal with that in 2.6.

> Where does this matter compared with Stage 1? In Stage 1 the agent could reach your whole computer.
> Now it can reach only the project folder — and whatever you install in the container.

## 2.2 Install Docker and the Dev Containers extension

1. Install [Docker Desktop](https://www.docker.com/products/docker-desktop/) (Windows, macOS or
   Linux) and start it. Wait until it says it is running.
2. In VS Code, open the Extensions panel and install **Dev Containers** (by Microsoft).

Check Docker from the terminal:

🖥️ Terminal
```bash
docker --version
docker run --rm hello-world
```

The second command downloads a tiny image, runs it as a container, prints *"Hello from Docker!"*
and removes the container. If you see that message, Docker works.

## 2.3 Describe the container in one file

VS Code needs a file that describes the container: which starting system, and which programs to add.
In your project folder create a folder called `.devcontainer` (with the dot), and inside it a file
`devcontainer.json`:

📄 File: `.devcontainer/devcontainer.json`
```json
{
  "name": "Agents demo",
  "image": "mcr.microsoft.com/devcontainers/python:3.12",

  "features": {
    "ghcr.io/devcontainers/features/node:1": {},
    "ghcr.io/anthropics/devcontainer-features/claude-code:1": {}
  },

  "postCreateCommand": "npm install -g @google/gemini-cli",

  "customizations": {
    "vscode": {
      "extensions": ["ms-python.python", "ms-toolsai.jupyter"]
    }
  }
}
```

That is the whole setup. Line by line:

| Line | What it means |
|---|---|
| `name` | A label shown in VS Code. Anything you like. |
| `image` | The starting point: a Linux system with Python 3.12 already installed, prepared by Microsoft. (This replaces step 1.2b.) |
| `features` | Ready-made add-ons, installed when the container is built. Here: **Node.js** (replaces 1.2c) and **Claude Code** (replaces 1.3). |
| `postCreateCommand` | A command run automatically after the container is created. Here it installs Gemini CLI — the same command you typed in 1.3. |
| `customizations` | VS Code extensions to install *inside* the container: Python and Jupyter, so this notebook works there too. |

Notice: this file is a written record of everything you did by hand in Stage 1. Send it to a
colleague and they get exactly the same environment.

> **Why is Gemini installed differently from Claude Code?** Anthropic publishes a ready-made
> *feature* for Claude Code; Google does not publish one for Gemini CLI, so we install it with an
> ordinary command. Both work.

## 2.4 Open the folder inside the container

1. Open the Command Palette: `Ctrl+Shift+P` (Windows/Linux) or `Cmd+Shift+P` (macOS).
2. Type and choose **Dev Containers: Reopen in Container**.
3. Wait. The first time takes several minutes, because Docker downloads the image and installs
   everything. Later openings are much faster.

When it finishes, the bottom-left corner of VS Code shows **Dev Container: Agents demo**. From now
on, every terminal you open in this window runs *inside the container*.

Reopen this notebook. When asked for a kernel, choose the Python inside the container
(`/usr/local/bin/python`). If VS Code offers to install `ipykernel`, accept.

Run the cell below. Compare its output with what you saw in Stage 1.

In [ ]:
import getpass, os, shutil
from pathlib import Path

print("user       :", getpass.getuser())
print("home folder:", Path.home())
print("this folder:", Path.cwd())
print("inside a container?", "yes" if Path("/.dockerenv").exists() or os.environ.get("REMOTE_CONTAINERS") else "no")
print()
for program in ("claude", "gemini", "node", "npm"):
    print(f"{program:7} ->", shutil.which(program) or "NOT FOUND")

You should see user **vscode**, home **/home/vscode**, and a folder under **/workspaces/**. These are
the container's, not yours: your personal files are not there.

The `structures/NiO_POSCAR` file from Stage 1 *is* still there, because it lives in the shared
project folder.

## 2.5 Sign in and repeat the task

The agents inside the container are new installations, so sign in once more:

🖥️ Terminal (inside the container)
```bash
claude
gemini
```

> **If the browser login finishes but the terminal keeps waiting:** the browser shows a code — copy
> it and paste it into the terminal where it says *Paste code here*. For Gemini you can instead use a
> free API key from [AI Studio](https://aistudio.google.com/apikey):
> `export GEMINI_API_KEY=your_key`

Now give the agent the **same request as in 1.5**. It should behave exactly as before and reach the
same answer (2.085 Å, 6 neighbours). The difference is not in what the agent does, but in what it
*could* touch if something went wrong.

Try it yourself: ask the agent *"List the files in my home folder."* It will show the container's
nearly empty home folder, not yours.

## 2.6 Keep your login after a rebuild *(recommended)*

Whenever you change `devcontainer.json`, VS Code must **rebuild** the container
(Command Palette → **Dev Containers: Rebuild Container**). A rebuild throws the old container away
— including your agent logins, which are saved in the home folder. You would have to sign in again
every time.

The fix is a **volume**: a storage space that Docker keeps *outside* the container, so it survives
rebuilds. We attach one volume where each agent stores its login. Replace your file with:

📄 File: `.devcontainer/devcontainer.json`
```json
{
  "name": "Agents demo",
  "image": "mcr.microsoft.com/devcontainers/python:3.12",

  "features": {
    "ghcr.io/devcontainers/features/node:1": {},
    "ghcr.io/anthropics/devcontainer-features/claude-code:1": {}
  },

  "containerEnv": {
    "CLAUDE_CONFIG_DIR": "/home/vscode/.claude"
  },

  "mounts": [
    "source=claude-login-${devcontainerId},target=/home/vscode/.claude,type=volume",
    "source=gemini-login-${devcontainerId},target=/home/vscode/.gemini,type=volume"
  ],

  "postCreateCommand": "sudo chown -R vscode:vscode /home/vscode/.claude /home/vscode/.gemini && npm install -g @google/gemini-cli",

  "customizations": {
    "vscode": {
      "extensions": ["ms-python.python", "ms-toolsai.jupyter"]
    }
  }
}
```

What changed:

| New line | What it does |
|---|---|
| `mounts` | Attaches two volumes: one at `~/.claude` (Claude's login) and one at `~/.gemini` (Gemini's login). `${devcontainerId}` is replaced by an ID unique to this project, so different projects don't share logins. |
| `containerEnv` → `CLAUDE_CONFIG_DIR` | Tells Claude Code to keep *all* its settings inside `~/.claude`. Without it, Claude also writes one file next to that folder, which would be lost. |
| `sudo chown -R vscode:vscode …` | New volumes belong to the administrator account (`root`), and you work as the ordinary user `vscode`. This command gives the folders to `vscode` so the agents are allowed to save your login there. (`sudo` = run as administrator, `chown` = change owner.) |

Rebuild the container, sign in one last time, then rebuild again: you should stay signed in.

In [ ]:
# Both login folders must be writable by you (user 'vscode').
from pathlib import Path

for folder in (Path.home() / ".claude", Path.home() / ".gemini"):
    if not folder.exists():
        print(f"{folder}: not created yet (appears after you sign in)")
        continue
    test = folder / ".write-test"
    try:
        test.touch(); test.unlink()
        print(f"{folder}: OK, writable")
    except PermissionError:
        print(f"{folder}: PERMISSION DENIED -> run: sudo chown -R vscode:vscode {folder}")

---
# Stage 3 — Next steps *(optional preview)*

## 3.1 Where a bare agent stops

Ask the agent something that isn't in your files:

```
What is the energy above hull of La3Ni2O7 in the Materials Project database,
and what is its Materials Project ID? Only answer from a source you can
actually query. If you cannot verify it, say so.
```

One of three things happens:

1. **It says it cannot check.** Honest — that's the behaviour you want.
2. **It answers from memory.** It may sound confident and may even be right, but you cannot verify
   it. Look: there is no tool call behind the answer.
3. **It searches the web** and reports what some page said. Better, but a web page is not the
   database.

The agent is not "not smart enough". It simply has **no connection** to the Materials Project.

## 3.2 Giving the agent new tools

To fix that, you give the agent new tools. The standard way to do this is called **MCP** (Model
Context Protocol). Think of it as a **universal plug**: someone writes a small program — an *MCP
server* — that offers a few tools, such as *"search the Materials Project"*, and any agent that
understands MCP (Claude Code, Gemini CLI, and others) can plug into it and use those tools.

With the Materials Project server connected, the same question returns a real Materials Project ID
and a number that came from the database. You can even write your own tools in Python — for
example, one that downloads a structure and prepares VASP input files.

That is the subject of the **advanced notebook** (`02_agents_with_tools_advanced.ipynb`), which
builds on the container you just made.

---
## Troubleshooting

| Problem | Likely cause | What to do |
|---|---|---|
| `command not found` right after installing something | The terminal was open before the install. | Close the terminal and open a new one. |
| `claude` / `gemini` not found on Windows | The installer's folder is not on PATH yet. | Restart VS Code. If it persists, re-run the installer. |
| `docker: permission denied` (Linux) | Your user is not in the `docker` group. | `sudo usermod -aG docker $USER`, then log out and back in. |
| *Reopen in Container* fails immediately | Docker Desktop is not running. | Start Docker Desktop, wait until it's ready, try again. |
| Notebook cells don't run inside the container | No kernel selected. | Click *Select Kernel* (top right) → the Python in the container; accept installing `ipykernel`. |
| Login finishes in the browser but the terminal keeps waiting | The browser can't reach the container. | Copy the code shown in the browser and paste it into the terminal. |
| `Permission denied` in `~/.claude` or `~/.gemini` | Volumes were created by `root`. | `sudo chown -R vscode:vscode ~/.claude ~/.gemini` (already in 2.6's file). |
| Logged out after every rebuild | No volumes. | Use the file from 2.6. |

---
## Glossary

| Term | Meaning |
|---|---|
| **AI model / LLM** | The AI itself. On its own it only reads and writes text. |
| **Agent** | An AI model connected to a computer so it can take actions (read files, run commands) and see the results. |
| **Tool / tool call** | One action the agent is allowed to take, e.g. "run this command". A tool call is one use of it. |
| **Terminal** | A window where you type commands. |
| **CLI** | *Command-line interface*: a program used by typing in a terminal. |
| **Node.js / npm** | Node.js runs JavaScript programs; npm installs them (like Python and pip). |
| **Kernel** | The Python that runs a notebook's code cells. |
| **Host** | Your real computer, as opposed to the container running on it. |
| **Docker** | The program that builds and runs containers. |
| **Image** | The recipe/starting point a container is made from. |
| **Container** | An isolated, disposable "computer inside your computer", made from an image. |
| **Dev container** | A container that VS Code opens your project in. Described by `.devcontainer/devcontainer.json`. |
| **Feature** | A ready-made add-on for a dev container that installs one program (e.g. Node.js). |
| **Rebuild** | Throw the container away and create it again from the (possibly changed) file. |
| **Volume** | Storage kept by Docker outside the container, so it survives rebuilds. |
| **Environment variable** | A named setting (e.g. `GEMINI_API_KEY`) that programs can read. Often used for secrets. |
| **API / API key** | An API is a way for programs to talk to a service; the key is your secret password for it. Never share or commit it. |
| **MCP / MCP server** | A standard "plug" for giving agents new tools; an MCP server is a program that provides such tools. |